# Bayesian logistic regression with PyMC

This notebook adds a Bayesian classifier for the observed death event. It uses only the 11 baseline predictors and excludes follow-up duration. Standardization is estimated separately within each outer training fold. A regularized logistic model produces posterior probabilities and uncertainty; its held-out predictions are evaluated with stratified 5-fold cross-validation. This is not the same estimand as the survival analysis.

## Model

For patient *i*, let `x_i` be the training-fold standardized predictors.

`y_i ~ Bernoulli(p_i)` and `logit(p_i) = alpha + x_i beta`.

Use `alpha ~ Normal(0, 1.5)` and `beta_j ~ Normal(0, 1)` as regularizing priors. The coefficients are not selected or tuned using held-out folds. Posterior probabilities for each held-out patient are averaged over posterior draws; 90% intervals summarize parameter uncertainty.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from scipy.special import expit
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from pro_gradu.classification import classification_features
from pro_gradu.data import load_dataset
from pro_gradu.evaluation import threshold_metrics

SEED = 42
DRAWS = 500
TUNE = 500
CHAINS = 2
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'heart_failure_clinical_records_dataset.csv').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
df = load_dataset(PROJECT_ROOT / 'heart_failure_clinical_records_dataset.csv')
X, y = classification_features(df)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'{len(df)} records; event prevalence {y.mean():.1%}; PyMC {pm.__version__}; ArviZ {az.__version__}')

In [ ]:
fold_rows = []
oof_rows = []
prior_event_rates = None

for fold, (train_pos, test_pos) in enumerate(outer_cv.split(X, y), start=1):
    scaler = StandardScaler().fit(X.iloc[train_pos])
    x_train = scaler.transform(X.iloc[train_pos])
    x_test = scaler.transform(X.iloc[test_pos])
    y_train = y.iloc[train_pos].to_numpy(dtype='int8')
    y_test = y.iloc[test_pos].to_numpy(dtype='int8')
    coords = {'feature': X.columns.to_list()}

    with pm.Model(coords=coords) as model:
        alpha = pm.Normal('alpha', mu=0, sigma=1.5)
        beta = pm.Normal('beta', mu=0, sigma=1, dims='feature')
        p_train = pm.Deterministic('p_train', pm.math.sigmoid(alpha + x_train @ beta))
        pm.Bernoulli('event', p=p_train, observed=y_train)
        if fold == 1:
            prior = pm.sample_prior_predictive(samples=500, random_seed=SEED)
            prior_event_rates = prior.prior_predictive['event'].mean(dim='event_dim_0').values.ravel()
        idata = pm.sample(
            draws=DRAWS, tune=TUNE, chains=CHAINS, cores=1, target_accept=0.9,
            random_seed=SEED + fold, progressbar=False, return_inferencedata=True,
        )

    posterior = idata.posterior
    alpha_draws = posterior['alpha'].values.reshape(-1)
    beta_draws = posterior['beta'].values.reshape(-1, X.shape[1])
    p_draws = expit(x_test @ beta_draws.T + alpha_draws)
    p_mean = p_draws.mean(axis=1)
    p_low, p_high = np.quantile(p_draws, [0.05, 0.95], axis=1)
    threshold = threshold_metrics(y_test, p_mean, threshold=0.5)
    diagnostics = az.summary(idata, var_names=['alpha', 'beta'], kind='diagnostics')
    fold_rows.append({
        'fold': fold, 'roc_auc': roc_auc_score(y_test, p_mean),
        'average_precision': average_precision_score(y_test, p_mean),
        'brier_score': brier_score_loss(y_test, p_mean),
        'log_loss': log_loss(y_test, p_mean, labels=[0, 1]),
        'observed_event_rate': y_test.mean(), 'mean_predicted_risk': p_mean.mean(),
        **threshold, 'max_rhat': diagnostics['r_hat'].max(),
        'min_ess_bulk': diagnostics['ess_bulk'].min(),
        'min_ess_tail': diagnostics['ess_tail'].min(),
    })
    for idx, observed, mean, low, high in zip(X.index[test_pos], y_test, p_mean, p_low, p_high):
        oof_rows.append({'index': idx, 'fold': fold, 'y_true': int(observed),
                         'p_mean': mean, 'p05': low, 'p95': high})

fold_results = pd.DataFrame(fold_rows)
oof = pd.DataFrame(oof_rows).set_index('index').sort_index()
fold_results

## Prior predictive behavior

The first outer training fold is used to standardize predictors before drawing from the stated prior. This checks the implied event rates before observing its training outcomes.

In [ ]:
pd.Series(prior_event_rates).describe(percentiles=[.05, .5, .95]).to_frame('prior_event_rate')

## Convergence and predictive checks

R-hat should be below 1.01 and bulk/tail effective sample sizes should be adequate (inspect all folds, not only the average). Predictive calibration is summarized by fold event prevalence, mean predicted risk, Brier score, and log loss. The 90% intervals describe posterior uncertainty for each held-out prediction; they do not include variation across new cohorts.

In [ ]:
diagnostic_columns = ['fold', 'max_rhat', 'min_ess_bulk', 'min_ess_tail']
fold_results[diagnostic_columns]

In [ ]:
fold_results[['fold', 'observed_event_rate', 'mean_predicted_risk', 'brier_score', 'log_loss']]

In [ ]:
fold_results[['roc_auc', 'average_precision', 'balanced_accuracy', 'sensitivity', 'specificity', 'accuracy']].agg(['mean', 'std']).T.round(3)

## Limitations

This regularized Bayesian logistic model estimates whether an event was observed during follow-up, not time to death. The cohort is small, the priors affect shrinkage, and internal cross-validation is not external validation. If diagnostics fail in any fold, increase tuning/draws or revise the model before interpreting the estimates. A 0.5 cutoff is shown for comparability and is not a clinical decision threshold.